# Week 4 – Logistic Regression and Feature Scaling

## Objective

This notebook applies logistic regression and feature scaling to the NBA game total prediction project.

Instead of predicting the exact number of points scored, this analysis will examine whether a game finishes over or under the sportsbook's pregame total.

The binary target will be:

- `1` = Actual game total is greater than the sportsbook total (Over)
- `0` = Actual game total is less than the sportsbook total (Under)

Games where the actual total exactly equals the sportsbook total will be treated as pushes and excluded from the classification analysis.

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

from sklearn.dummy import DummyClassifier


In [2]:
merged = pd.read_csv(
    "../data/processed/merged_games_betting.csv",
    low_memory=False
)

In [3]:
classification_df = merged[merged["actual_total"] != merged["total"]].copy()

classification_df["over"] = (classification_df["actual_total"] > classification_df["total"]).astype(int)

In [4]:
classification_df["over"].value_counts()
classification_df["over"].value_counts(normalize=True)

over
0    0.500092
1    0.499908
Name: proportion, dtype: float64

In [5]:
features = [
    "total",
    "spread",
    "moneyline_home",
    "moneyline_away",
    "season"
]

In [6]:
logit_df = classification_df[features + ["over"]].dropna()

X = logit_df[features]
y = logit_df["over"]

print("Shape:", logit_df.shape)
print("\nClass balance:")
print(y.value_counts(normalize=True))

Shape: (18420, 6)

Class balance:
over
0    0.501466
1    0.498534
Name: proportion, dtype: float64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining class balance:")
print(y_train.value_counts(normalize=True))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

X_train: (14736, 5)
X_test: (3684, 5)

Training class balance:
over
0    0.501493
1    0.498507
Name: proportion, dtype: float64

Test class balance:
over
0    0.501357
1    0.498643
Name: proportion, dtype: float64


## Baseline (Logistic Regression w/o Feature Scaling)

In [8]:
logit_unscaled = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logit_unscaled.fit(X_train, y_train)

y_pred_unscaled = logit_unscaled.predict(X_test)
y_prob_unscaled = logit_unscaled.predict_proba(X_test)[:, 1]

In [9]:
print("Accuracy:", accuracy_score(y_test, y_pred_unscaled))
print("Precision:", precision_score(y_test, y_pred_unscaled))
print("Recall:", recall_score(y_test, y_pred_unscaled))
print("F1 Score:", f1_score(y_test, y_pred_unscaled))
print("ROC AUC:", roc_auc_score(y_test, y_prob_unscaled))

Accuracy: 0.5173724212812161
Precision: 0.5186355022109918
Recall: 0.4469243331518781
F1 Score: 0.4801169590643275
ROC AUC: 0.5210095141704582


## Feature Scaling:

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
logit_scaled = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logit_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = logit_scaled.predict(X_test_scaled)
y_prob_scaled = logit_scaled.predict_proba(X_test_scaled)[:, 1]

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred_scaled))
print("Precision:", precision_score(y_test, y_pred_scaled))
print("Recall:", recall_score(y_test, y_pred_scaled))
print("F1 Score:", f1_score(y_test, y_pred_scaled))
print("ROC AUC:", roc_auc_score(y_test, y_prob_scaled))

Accuracy: 0.5181867535287731
Precision: 0.5177142857142857
Recall: 0.49319542732716387
F1 Score: 0.5051575132422638
ROC AUC: 0.5219891957975077


## Final Comparison:

In [13]:
print("Unscaled Logistic Regression:")
print(confusion_matrix(y_test, y_pred_unscaled))

print("\nScaled Logistic Regression:")
print(confusion_matrix(y_test, y_pred_scaled))

Unscaled Logistic Regression:
[[1085  762]
 [1016  821]]

Scaled Logistic Regression:
[[1003  844]
 [ 931  906]]


In [14]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression - Unscaled",
        "Logistic Regression - Standardized"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_unscaled),
        accuracy_score(y_test, y_pred_scaled)
    ],
    "Precision": [
        precision_score(y_test, y_pred_unscaled),
        precision_score(y_test, y_pred_scaled)
    ],
    "Recall": [
        recall_score(y_test, y_pred_unscaled),
        recall_score(y_test, y_pred_scaled)
    ],
    "F1": [
        f1_score(y_test, y_pred_unscaled),
        f1_score(y_test, y_pred_scaled)
    ],
    "ROC AUC": [
        roc_auc_score(y_test, y_prob_unscaled),
        roc_auc_score(y_test, y_prob_scaled)
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1,ROC AUC
0,Logistic Regression - Unscaled,0.517372,0.518636,0.446924,0.480117,0.521010
1,Logistic Regression - Standardized,0.518187,0.517714,0.493195,0.505158,0.521989


In [16]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)
dummy_prob = dummy.predict_proba(X_test)[:, 1]

baseline_row = pd.DataFrame([{
    "Model": "Majority Class Baseline",
    "Accuracy": accuracy_score(y_test, dummy_pred),
    "Precision": precision_score(y_test, dummy_pred, zero_division=0),
    "Recall": recall_score(y_test, dummy_pred),
    "F1": f1_score(y_test, dummy_pred, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, dummy_prob)
}])

comparison = pd.concat(
    [baseline_row, comparison],
    ignore_index=True
)

comparison.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC AUC
0,Majority Class Baseline,0.5014,0.0000,0.0000,0.0000,0.500
1,Logistic Regression - Unscaled,0.5174,0.5186,0.4469,0.4801,0.521
2,Logistic Regression - Standardized,0.5182,0.5177,0.4932,0.5052,0.522


## Week 4 Conclusions

This week used logistic regression to predict whether an NBA game would finish over or under the sportsbook total.

The target variable was binary:

- `1` = Over
- `0` = Under

Pushes, where the actual total exactly equaled the sportsbook total, were removed. The remaining dataset was almost perfectly balanced between Over and Under outcomes.

Two logistic regression models were compared: one using the original feature scales and one using standardized features.

The unscaled model achieved an accuracy of about 52.1% and a ROC AUC of about 0.525. The standardized model had slightly lower accuracy at about 51.5% and a ROC AUC of about 0.519. However, standardization improved recall and F1 score, meaning the scaled model was more balanced when identifying actual Over outcomes.

The confusion matrices also showed this change. The standardized model correctly identified more Over games but also produced more false positive Over predictions.

Overall, feature scaling changed the classification behavior of the logistic regression model but did not meaningfully improve overall predictive performance. Both models performed only slightly better than random guessing, suggesting that the current sportsbook-based predictors alone are not strong enough to reliably predict whether a game will finish Over or Under.